# 05 · Explainability (SHAP) — gated

SHAP is computed **only** for a versioned artifact whose `metadata.json` has `"evaluated": true`.
Attributions describe the model, not physical causes. Never present them as "why the air is bad".

In [ ]:
from pavana_ml.models.lgbm import latest_evaluated_artifact

artifact = latest_evaluated_artifact()
print("evaluated artifact:", artifact)

In [ ]:
if artifact is not None:
    import json

    import lightgbm as lgb
    import pandas as pd
    import shap

    from pavana_ml.paths import PROCESSED_DIR

    meta = json.loads((artifact / "metadata.json").read_text())
    model = lgb.Booster(model_file=str(artifact / "model.txt"))
    X = (
        pd.read_parquet(PROCESSED_DIR / "model_table.parquet")[meta["features"]]
        .dropna()
        .sample(2000, random_state=0)
    )
    shap.summary_plot(shap.TreeExplainer(model).shap_values(X), X)
else:
    print("No evaluated model artifact — nothing to explain. (This is expected for now.)")